# Slang Generation Demo

This demo will illustrate the basic workflow of the slang generation code accompanying the TACL paper *[A Computational Framework for Slang Generation](https://direct.mit.edu/tacl/article/doi/10.1162/tacl_a_00378/100687/A-Computational-Framework-for-Slang-Generation)* using slang definition data from Urban Dictionary (UD) and conventional definition data from WordNet.

To run this tutorial, you will need the following dependencies:

- Python 3
- Numpy
- Scipy
- tqdm
- NLTK
- Gensim
- PyTorch (torch)
- SBERT (sentence_transformers)
- [CatGO](https://github.com/zhewei-sun/CatGO)


In [ ]:
import numpy as np
import torch
import shutil
import copy

We first create a symbolic link pointing to the library. You will need to change the destination if your code sits in a different directory. 

In [ ]:
! ln -s ../Code slanggen

CatGO is a library that optimizes and runs models of categorization and can be obtained [here](https://github.com/zhewei-sun/CatGO). Once you have downloaded the code, please link it by replacing the target directory of the simlink below. 

In [ ]:
! ln -s ../../CatGO CatGO
import nltk
nltk.download('stopwords')

In [ ]:
from slanggen.util import *
from slanggen.dataloader import WN_Dataset, Urban_Dataset, OSD_Dataset, ZH_Dataset
from slanggen.encoder import FTEncoder, FTCachedEncoder, SBertEncoder, SenseEncoder, dump_vanilla_embeddings
from slanggen.contrastive import SlangGenTrainer
from slanggen.model import SlangGenModel

Specify a PyTorch device if necessary:

In [ ]:
torch.cuda.set_device(0)

Load conventional definition data using the builtin dataloaders. The *.npy* file loaded below contains a pre-processed version of WordNet definition sentences for all words that appear in both WordNet and UD.

In [ ]:
# wn_data = WN_Dataset('mix_conv_data_all_chime.npy') 
wn_data = WN_Dataset('mix_multilingual_conv_data_all_OD_chime.npy')
# wn_data = WN_Dataset('OD_OSD.npy')

print(wn_data)

# EN conv dataset, used for prediction test
en_conv_data = WN_Dataset('OD_OSD.npy')
print(en_conv_data)

# ZH conv dataset, used for prediction test
# zh_conv_data = WN_Dataset('ZH_conv_data.npy')
# print(zh_conv_data)
ZH_zh_conv_data = WN_Dataset('ZH_zh_conv_data_chime.npy')
print(ZH_zh_conv_data)

# RU conv dataset, used for prediction test
# ru_conv_data = WN_Dataset('RU_conv_data.npy')
# print(ru_conv_data)
RU_ru_conv_data = WN_Dataset('RU_ru_conv_data.npy')
print(RU_ru_conv_data)

Load slang definition data. The *.npy* file loaded below is a pre-processed version of the data released in this repository.

In [ ]:
# OSD_wn_data = OSD_Dataset('OSD_data.npy', wn_data)
# print(OSD_wn_data)

# We also want to try mixed data (EN + ZH) / (EN + RU)

# mix_data = OSD_Dataset('mix_slang_data_all_chime.npy', wn_data)
# mix_data = OSD_Dataset('RU_ru_slang_data.npy', wn_data)
mix_data = OSD_Dataset('mix_multilingual_slang_data_all_OD_chime.npy', wn_data)

print(mix_data)

# EN/ZH slang dataset, used for prediction test
en_slang_data = OSD_Dataset('OSD_V2.npy', en_conv_data)
# zh_slang_data = ZH_Dataset('ZH_slang_data.npy', zh_conv_data)
# ru_slang_data = ZH_Dataset('RU_slang_data.npy', ru_conv_data)

# Test the performance on other languages
zh_data = ZH_Dataset('ZH_zh_slang_data_chime.npy', ZH_zh_conv_data)
print(zh_data)
ru_data = ZH_Dataset('RU_ru_slang_data.npy', RU_ru_conv_data)
print(ru_data)

If you wish to use your own dataset, please create a dataloader object inheriting either *ConvDataset* or *SlangDataset* abstract classes found in *dataloader.py* and following the example data specifications in *dataloader.WN_Dataset* and *dataloader.Urban_Dataset*.

Now let's create a directory to store our results and load in some pre-generated data indices for train-test split:

In [ ]:
shutil.rmtree('Results')  
create_directory('Results')

In [ ]:
out_dir='Results/'

# dataset_osd = OSD_wn_data
# slang_inds_osd = DataIndex(np.load('train_ind_osd.npy'), np.load('dev_ind_osd.npy'), np.load('test_ind_osd.npy'))

dataset_mix = mix_data
# slang_inds_mix = DataIndex(np.load('train_ind_mix_all_chime.npy'), np.load('dev_ind_mix_all_chime.npy'), np.load('test_ind_mix_all_chime.npy'))
slang_inds_mix = DataIndex(np.load('train_ind_mix_multilingual_all_OD_chime.npy'), np.load('dev_ind_mix_multilingual_all_OD_chime.npy'), np.load('test_ind_mix_multilingual_all_OD_chime.npy'))

# For prediction tests
dataset_mix_en = en_slang_data
slang_inds_en = DataIndex(np.load('train_ind_OD_OSD.npy'), np.load('dev_ind_OD_OSD.npy'), np.load('test_ind_OD_OSD.npy'))

# dataset_mix_zh = zh_slang_data
# slang_inds_zh = DataIndex(np.load('train_ind_zh.npy'), np.load('dev_ind_zh.npy'), np.load('test_ind_zh.npy'))

# dataset_mix_ru = ru_slang_data
# slang_inds_ru = DataIndex(np.load('train_ind_ru.npy'), np.load('dev_ind_ru.npy'), np.load('test_ind_ru.npy'))

#Test the performance on other languages
dataset_zh_zh = zh_data
slang_inds_zh_zh = DataIndex(np.load('train_ind_ZH_zh_chime.npy'), np.load('dev_ind_ZH_zh_chime.npy'), np.load('test_ind_ZH_zh_chime.npy'))

dataset_ru_ru = ru_data
slang_inds_ru_ru = DataIndex(np.load('train_ind_ru_ru.npy'), np.load('dev_ind_ru_ru.npy'), np.load('test_ind_ru_ru.npy'))

The following encoder objects initializes a fastText encoder used for collaborative filtering. *FTEncoder* can be used to read in the original fastText embedding file. For efficiency, we have cached the words we need and use a cached encoder instead.

ft_encoder = FTEncoder('path to crawl-300d-2M-subword.vec')

In [ ]:
ft_encoder = FTCachedEncoder('ft_embed_cache_Urban.pickle')

The following commands sets up the contrastive trainer and the slang generation model:

In [ ]:
trainer_mix = SlangGenTrainer(dataset_mix, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)

You can modify the 'embed_name' param below to choose the sense encoding model, here is a list of supported models:
- bert-base-nli-mean-tokens -> 'SBERT_contrastive' (default)
- sentence-t5-base -> 'SBERT_t5'
- paraphrase-multilingual-MiniLM-L12-v2 -> 'SBERT-multilingual-MiniLM-L12-v2'
- LaBSE -> 'SBERT_LaBSE'
- paraphrase-multilingual-mpnet-base-v2 -> 'SBERT_mpnet'
- intfloat/multilingual-e5-base -> 'SBERT_e5_base'
- intfloat/multilingual-e5-large -> 'SBERT_e5_large'

In [ ]:
model_mix = SlangGenModel(trainer_mix, data_dir=out_dir, embed_name='SBERT_mpnet')

params = {'embed_name':'SBERT_mpnet', 'out_name':'predictions', 'model':'cf_prototype_5', 'prior':None, 'prior_name':'uniform', 'contr_params':None}

Invoke *model.train_contrastive* to train the contrastively learned sense embedding model:

Note: you can set mode to 'head' to train only the triplet head, or 'whole' to train both the sense encoding and the head. Also it's optional to change the fold_name if using a new dataset.

In [ ]:
model_mix.train_contrastive(slang_inds_mix, fold_name='mix_wn', params=params, mode='head')

In [ ]:
# Now we want to see how the mixed dataset performs on each language

# Step 1: Create symbolic links
!mkdir -p Results/mix_wn_en/SBERT_data
# !mkdir -p Results/mix_wn_zh/SBERT_data
# !mkdir -p Results/mix_wn_ru/SBERT_data

!cp Results/mix_wn/SBERT_data/SBERT_mpnet_with_head.pt Results/mix_wn_en/SBERT_data/
# !cp Results/mix_wn/SBERT_data/SBERT_mpnet_with_head.pt Results/mix_wn_ru/SBERT_data/

# !cp Results/mix_wn/SBERT_data/SBERT_contrastive_with_head.pt Results/mix_wn_en/SBERT_data/
# !cp Results/mix_wn/SBERT_data/SBERT_contrastive_with_head.pt Results/mix_wn_zh/SBERT_data/

# !cp Results/mix_wn/SBERT_data/SBERT_contrastive_whole_finetuned.pt Results/mix_wn_en/SBERT_data/
# !cp Results/mix_wn/SBERT_data/SBERT_contrastive_whole_finetuned.pt Results/mix_wn_zh/SBERT_data/

params['embed_name'] = 'SBERT_mpnet' 

trainer_mix_en = SlangGenTrainer(dataset_mix_en, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_mix_en   = SlangGenModel(trainer_mix_en, data_dir=out_dir, embed_name='SBERT_mpnet')

# trainer_mix_ru = SlangGenTrainer(dataset_mix_ru, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
# model_mix_ru   = SlangGenModel(trainer_mix_ru, data_dir=out_dir, embed_name='SBERT_mpnet')

In [ ]:
trainer_mix_en.get_trained_embeddings(slang_inds_en, fold_name='mix_wn_en', model_path='SBERT_mpnet')
# trainer_mix_zh.get_trained_embeddings(slang_inds_zh, fold_name='mix_wn_zh', model_path='SBERT_mpnet')
# trainer_mix_ru.get_trained_embeddings(slang_inds_ru, fold_name='mix_wn_ru', model_path='SBERT_mpnet')

model_mix_en.train_categorization(slang_inds_en, fold_name='mix_wn_en', params=params)
results_mix_en = model_mix_en.get_results(fold_name='mix_wn_en', mode='train', params=params)

# params_zh = {'embed_name':'SBERT_mpnet', 'out_name':'predictions', 'model':'prototype', 'prior':None, 'prior_name':'uniform', 'contr_params':None}
# params_ru = {'embed_name':'SBERT_mpnet', 'out_name':'predictions', 'model':'prototype', 'prior':None, 'prior_name':'uniform', 'contr_params':None}

# model_mix_zh.train_categorization(slang_inds_zh, fold_name='mix_wn_zh', params=params_zh)
# results_mix_zh = model_mix_zh.get_results(fold_name='mix_wn_zh', mode='train', params=params_zh)

# model_mix_ru.train_categorization(slang_inds_ru, fold_name='mix_wn_ru', params=params_ru)
# results_mix_ru = model_mix_ru.get_results(fold_name='mix_wn_ru', mode='train', params=params_ru)

In [ ]:
print('Performance on train split - EN')
N_train_dev_mix_en = dataset_mix_en.N_total - slang_inds_en.test.shape[0]
train_rankings_mix_en = get_rankings(results_mix_en, np.arange(N_train_dev_mix_en), dataset_mix_en.vocab_ids[np.concatenate(slang_inds_en)])
np.mean(get_roc(train_rankings_mix_en, dataset_mix_en.V))

In [ ]:
# print('Performance on train split - ZH')
# N_train_dev_mix_zh = dataset_mix_zh.N_total - slang_inds_zh.test.shape[0]
# train_rankings_mix_zh = get_rankings(results_mix_zh, np.arange(N_train_dev_mix_zh), dataset_mix_zh.vocab_ids[np.concatenate(slang_inds_zh)])
# np.mean(get_roc(train_rankings_mix_zh, dataset_mix_zh.V))

In [ ]:
# print('Performance on train split - RU')
# N_train_dev_mix_ru = dataset_mix_ru.N_total - slang_inds_ru.test.shape[0]
# train_rankings_mix_ru = get_rankings(results_mix_ru, np.arange(N_train_dev_mix_ru), dataset_mix_ru.vocab_ids[np.concatenate(slang_inds_ru)])
# np.mean(get_roc(train_rankings_mix_ru, dataset_mix_ru.V))

In [ ]:
model_mix_en.predict_testset(slang_inds_en, fold_name='mix_wn_en', params=params)
results_mix_en = model_mix_en.get_results(fold_name='mix_wn_en', mode='test', params=params)

# model_mix_zh.predict_testset(slang_inds_zh, fold_name='mix_wn_zh', params=params_zh)
# results_mix_zh = model_mix_zh.get_results(fold_name='mix_wn_zh', mode='test', params=params_zh)

# model_mix_ru.predict_testset(slang_inds_ru, fold_name='mix_wn_ru', params=params_ru)
# results_mix_ru = model_mix_ru.get_results(fold_name='mix_wn_ru', mode='test', params=params_ru)

In [ ]:
# print('Performance on test split - EN')
# N_train_dev_mix_en = dataset_mix_en.N_total - slang_inds_en.test.shape[0]
# test_rankings_mix_en = get_rankings(results_mix_en, np.arange(N_train_dev_mix_en, dataset_mix_en.N_total), dataset_mix_en.vocab_ids[np.concatenate(slang_inds_en)])
# np.mean(get_roc(test_rankings_mix_en, dataset_mix_en.V))

print('Performance on test split - EN')
inds = slang_inds_en.test                      
labels = dataset_mix_en.vocab_ids                 
test_rankings_mix_en = get_rankings(results_mix_en, inds, labels)
np.mean(get_roc(test_rankings_mix_en, dataset_mix_en.V))

In [ ]:


# print('Performance on test split - ZH')
# inds = slang_inds_zh.test                      
# labels = dataset_mix_zh.vocab_ids                 
# test_rankings_mix_zh = get_rankings(results_mix_zh, inds, labels)
# np.mean(get_roc(test_rankings_mix_zh, dataset_mix_zh.V))

In [ ]:


# print('Performance on test split - RU')
# inds = slang_inds_ru.test                      
# labels = dataset_mix_ru.vocab_ids                 
# test_rankings_mix_ru = get_rankings(results_mix_ru, inds, labels)
# np.mean(get_roc(test_rankings_mix_ru, dataset_mix_ru.V))

In [ ]:
# Test how the model perform on different datasets in their own languages
params['embed_name'] = 'SBERT_mpnet' 

!mkdir -p Results/test_zh/SBERT_data
!mkdir -p Results/test_ru/SBERT_data

!cp Results/mix_wn/SBERT_data/SBERT_mpnet_with_head.pt Results/test_zh/SBERT_data/
# !cp Results/osd_wn/SBERT_data/SBERT_mpnet_whole_finetuned.pt Results/test_zh/SBERT_data/

trainer_test_zh = SlangGenTrainer(zh_data, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_test_zh   = SlangGenModel(trainer_test_zh, data_dir=out_dir, embed_name='SBERT_mpnet')

trainer_test_zh.get_trained_embeddings(slang_inds_zh_zh, fold_name='test_zh', model_path='SBERT_mpnet')

model_test_zh.train_categorization(slang_inds_zh_zh, fold_name='test_zh', params=params)
results_zh = model_test_zh.get_results(fold_name='test_zh', mode='train', params=params)

print('Performance on train split - ZH')
N_train_dev_zh = zh_data.N_total - slang_inds_zh_zh.test.shape[0]
train_rankings_zh = get_rankings(results_zh, np.arange(N_train_dev_zh), zh_data.vocab_ids[np.concatenate(slang_inds_zh_zh)])
np.mean(get_roc(train_rankings_zh, zh_data.V))

In [ ]:
model_test_zh.predict_testset(slang_inds_zh_zh, fold_name='test_zh', params=params)
results_zh = model_test_zh.get_results(fold_name='test_zh', mode='test', params=params)

print('Performance on test split - ZH')
inds = slang_inds_zh_zh.test                      
labels = zh_data.vocab_ids                 
test_rankings_zh = get_rankings(results_zh, inds, labels)
np.mean(get_roc(test_rankings_zh, zh_data.V))

In [ ]:
!cp Results/mix_wn/SBERT_data/SBERT_mpnet_with_head.pt Results/test_ru/SBERT_data/
# !cp Results/osd_wn/SBERT_data/SBERT_mpnet_whole_finetuned.pt Results/test_ru/SBERT_data/

trainer_test_ru = SlangGenTrainer(ru_data, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_test_ru   = SlangGenModel(trainer_test_ru, data_dir=out_dir, embed_name='SBERT_mpnet')

trainer_test_ru.get_trained_embeddings(slang_inds_ru_ru, fold_name='test_ru', model_path='SBERT_mpnet')

model_test_ru.train_categorization(slang_inds_ru_ru, fold_name='test_ru', params=params)
results_ru = model_test_ru.get_results(fold_name='test_ru', mode='train', params=params)

print('Performance on train split - RU')
N_train_dev_ru = ru_data.N_total - slang_inds_ru_ru.test.shape[0]
train_rankings_ru = get_rankings(results_ru, np.arange(N_train_dev_ru), ru_data.vocab_ids[np.concatenate(slang_inds_ru_ru)])
np.mean(get_roc(train_rankings_ru, ru_data.V))

In [ ]:
model_test_ru.predict_testset(slang_inds_ru_ru, fold_name='test_ru', params=params)
results_ru = model_test_ru.get_results(fold_name='test_ru', mode='test', params=params)

print('Performance on test split - RU')
inds = slang_inds_ru_ru.test                      
labels = ru_data.vocab_ids                 
test_rankings_ru = get_rankings(results_ru, inds, labels)
np.mean(get_roc(test_rankings_ru, ru_data.V))